In [28]:
import pandas as pd 
import numpy as np

In [29]:
movies=pd.read_csv(r"E:\Movie_Rate_DB\movies.csv")
ratings=pd.read_csv(r"E:\Movie_Rate_DB\ratings.csv")
tags=pd.read_csv(r"E:\Movie_Rate_DB\tags.csv")

In [30]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [31]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [32]:
tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


# Collaborative Filtering

In [33]:
user_movie_ratings = ratings.pivot(index='userId', columns='movieId', values='rating')


In [34]:
user_movie_ratings.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
user_movie_ratings.fillna(0, inplace=True)

In [36]:
from sklearn.decomposition import TruncatedSVD

svd=TruncatedSVD(n_components=50, random_state=11,n_iter=10)
svd_matrix=svd.fit(user_movie_ratings)
svd_matrix

TruncatedSVD(n_components=50, n_iter=10, random_state=11)

In [37]:
import os
model_dir = '../models'

# Check if the directory exists, and create it if not
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

In [38]:
import joblib

joblib.dump(svd, os.path.join(model_dir, 'svd_model.pkl'))

svd = joblib.load('../models/svd_model.pkl')


In [39]:
svd.components_.shape

(50, 9724)

In [40]:
svd.components_[:5]

array([[ 7.04498985e-02,  3.85393459e-02,  1.59129220e-02, ...,
         6.46836073e-05,  6.46836073e-05,  2.71729303e-04],
       [-2.75911949e-02, -2.06662722e-03, -2.47146155e-02, ...,
         5.97586244e-04,  5.97586244e-04,  1.27236200e-03],
       [ 7.84438842e-02,  5.68447103e-02,  1.80051145e-02, ...,
        -8.71093880e-05, -8.71093880e-05,  1.22833344e-04],
       [ 1.68374063e-03, -5.06728558e-02, -3.74528571e-02, ...,
         4.90112264e-05,  4.90112264e-05,  8.94587289e-04],
       [-1.22103596e-02, -1.98793412e-02,  1.56337722e-04, ...,
         1.40608864e-04,  1.40608864e-04, -7.26189681e-04]])

In [41]:
def recommend_content(movie_id, svd, top_n=10):

    movie_idx = movie_id - 1
    movie_vec = svd.components_[:, movie_idx]

    similarity_scores = svd.components_.T.dot(movie_vec) # ( 9724 x 50 ) x (50 x 1) = (9724 x 1)

    top_movie_indices = similarity_scores.argsort()[-top_n-1:-1][::-1]

    recommended_titles = [movies.iloc[i]['title'] for i in top_movie_indices if i != movie_idx]

    return recommended_titles


In [42]:
# Example: Recommend top 10 movies similar to movie with movie_id = 1
top_recommendations=recommend_content(movie_id=1,svd=svd, top_n=10)

# Display recommended movies
for i in top_recommendations:
    print(i)


Independence Day (a.k.a. ID4) (1996)
Willy Wonka & the Chocolate Factory (1971)
Mission: Impossible (1996)
Twister (1996)
'night Mother (1986)
High Heels and Low Lifes (2001)
Jurassic Park (1993)
Rififi (Du rififi chez les hommes) (1955)
Spanish Apartment, The (L'auberge espagnole) (2002)
Star Wars: Episode IV - A New Hope (1977)


# Content Based Filtering

In [43]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [44]:
movies.movieId.count()

9742

In [45]:
mean_count_rating=ratings.groupby('movieId').agg(
    {
        "rating": ["mean", "count"],
    }
)

In [46]:
mean_count_rating.head()

rating      
             mean count
movieId                
1        3.920930   215
2        3.431818   110
3        3.259615    52
4        2.357143     7
5        3.071429    49

In [47]:
mean_count_rating.columns

MultiIndex([('rating',  'mean'),
            ('rating', 'count')],
           )

In [48]:
mean_count_rating.columns = ['avg_rating', 'rating_count']
print("After renaming:", mean_count_rating.columns.tolist())
mean_count_rating.reset_index(inplace=True)
mean_count_rating.head()

After renaming: ['avg_rating', 'rating_count']


,movieId,avg_rating,rating_count
0,1,3.920930,215
1,2,3.431818,110
2,3,3.259615,52
3,4,2.357143,7
4,5,3.071429,49


In [49]:
movies = movies.merge(mean_count_rating, on='movieId', how='left')
movies.tail()

,movieId,title,genres,avg_rating,rating_count
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy,4.0,1.0
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy,3.5,1.0
9739,193585,Flint (2017),Drama,3.5,1.0
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation,3.5,1.0
9741,193609,Andrew Dice Clay: Dice Rules (1991),Comedy,4.0,1.0


In [50]:
movies.genres = movies.genres.str.split('|')
def movie_genre(x):
    return " ".join(x)
movies.genres = movies.genres.apply(movie_genre)

In [51]:
movies['text_features'] = movies['title'] + ' ' + movies['genres']
movies.head()

,movieId,title,genres,avg_rating,rating_count,text_features
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,3.920930,215.0,Toy Story (1995) Adventure Animation Children ...
1,2,Jumanji (1995),Adventure Children Fantasy,3.431818,110.0,Jumanji (1995) Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance,3.259615,52.0,Grumpier Old Men (1995) Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance,2.357143,7.0,Waiting to Exhale (1995) Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,3.071429,49.0,Father of the Bride Part II (1995) Comedy


In [52]:
movies.isnull().sum()

movieId           0
title             0
genres            0
avg_rating       18
rating_count     18
text_features     0
dtype: int64

In [53]:
movies.dropna(axis=0,inplace=True)

In [54]:
movies.isnull().sum()

movieId          0
title            0
genres           0
avg_rating       0
rating_count     0
text_features    0
dtype: int64

In [55]:
movies.shape

(9724, 6)

In [56]:
numeric_features = movies[['avg_rating', 'rating_count']].fillna(0)

In [57]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf=TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['text_features'])


In [58]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
numeric_matrix = scaler.fit_transform(numeric_features)

In [59]:
from scipy.sparse import hstack

combined_matrix = hstack([tfidf_matrix, numeric_matrix])
combined_matrix.shape

(9724, 9262)

In [60]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim=cosine_similarity(combined_matrix)

In [61]:
np.save('../models/cosine_sim.npy', cosine_sim)

cosine_sim = np.load('../models/cosine_sim.npy')

In [62]:
def recommend_collab(movie_id, cosine_sim, top_n=10):
    movie_idx = movie_id - 1  # Assuming movie_id starts from 1
    
    similarity_scores = cosine_sim[movie_idx]
    similar_movie_indices = similarity_scores.argsort()[-top_n-1:-1][::-1]
    recommended_titles = [movies.iloc[i]['title'] for i in similar_movie_indices]
    
    return recommended_titles


In [63]:
recommended_movies = recommend_collab(movie_id=1, cosine_sim=cosine_sim, top_n=10)
for i in recommended_movies:
    print(i)

Toy Story 2 (1999)
Toy Story 3 (2010)
Shrek (2001)
Jumanji (1995)
Aladdin (1992)
Monsters, Inc. (2001)
Braveheart (1995)
Up (2009)
Lion King, The (1994)
Christmas Story, A (1983)


# Hybrid System Building

In [64]:
def hybrid_recommend(movie_id,svd,cosine_sim,top_n=10,content_weight=0.5,collab_weight=0.5):

    movie_idx=movie_id-1

    
    movie_vec=svd.components_[:, movie_idx]
    collab_score=svd.components_.T.dot(movie_vec)
    collab_score = collab_score.flatten()


    content_score=cosine_sim[movie_idx]
    content_score = content_score.flatten()

    hybrid_score=content_weight*content_score + collab_weight*collab_score

    top_movies_indecies=hybrid_score.argsort()[-top_n-1:-1][::-1]
    recommended_titles = [movies.iloc[i]['title'] for i in top_movies_indecies if i != movie_idx]
    recommended_genre = [movies.iloc[i]['genres'] for i in top_movies_indecies if i != movie_idx]
    return recommended_titles,recommended_genre

    

In [65]:
movies[movies['movieId'] == 1500]

,movieId,title,genres,avg_rating,rating_count,text_features
1146,1500,Grosse Pointe Blank (1997),Comedy Crime Romance,3.610169,59.0,Grosse Pointe Blank (1997) Comedy Crime Romance


In [66]:
hybrid_recommendations,hybrid_genre= hybrid_recommend(movie_id=1500,svd=svd,cosine_sim=cosine_sim,top_n=10,content_weight=0.5,collab_weight=0.5)

for i,j in zip(hybrid_recommendations,hybrid_genre):
    print(f"Movie: {i},          Genre: {j}")

Movie: Lolita (1962),          Genre: Drama Romance
Movie: Afterglow (1997),          Genre: Drama Romance
Movie: The Thief (1997),          Genre: Drama
Movie: Touch (1997),          Genre: Drama Fantasy Romance
Movie: Déjà Vu (1997),          Genre: Drama Romance
Movie: Assignment, The (1997),          Genre: Action Thriller
Movie: Love Jones (1997),          Genre: Romance
Movie: Brother (Brat) (1997),          Genre: Crime Drama
Movie: Wings of the Dove, The (1997),          Genre: Drama Romance
Movie: Romance (1999),          Genre: Drama Romance


In [67]:
movies.to_csv('../datasets/movies.csv', index=False)